# [Optional] Jupyter Notebook 101

**Goal**: About 15 minutes to get comfortable with the notebook environment so the rest of the workshop is about Bedrock, not the editor.

If you've used Jupyter before — run the cells and skip to [Lab 1: Prompts 101](./01-prompts-101.ipynb).


## Learning Objectives

By the end of this notebook, you'll have:

1. Selected **Bedrock Workshop (Python 3.13)** as the kernel
2. Run a Markdown cell vs a code cell — and seen the difference
3. Defined a variable in one cell, modified it in another — observed how state persists
4. Inspected the kernel's packages and learned how `%pip` differs from a shell command with `!`
5. Made a Bedrock call and read the response
6. Practiced restarting and interrupting the kernel when state gets confusing

A notebook combines explanations, code, and results. The **kernel** is the Python process that runs your code and remembers its variables.

---


### Select the Python 3.13 kernel

In Workshop Studio, select **Bedrock Workshop (Python 3.13)** at the top of the notebook. The environment, dependencies, and shared workshop infrastructure are already prepared. The next cell prints the interpreter and installed SDK versions so you can recognize the environment you are using.

If you are preparing a local notebook environment, run these commands from the repository root:

```bash
uv sync --frozen --python 3.13 --extra notebook --extra langfuse
uv run --no-sync python -m ipykernel install --user --name bedrock-workshop-py313 --display-name "Bedrock Workshop (Python 3.13)"
```

The lock file gives everyone the same package versions. The observability lesson shows how to inspect requests, token usage and results in CloudWatch or Langfuse.


In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Select Bedrock Workshop (Python 3.13), then restart the kernel.")

REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "workshop_utils").is_dir()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook inside the workshop repository.")

print("Kernel:", sys.executable)
print("Python:", sys.version.split()[0])
print({name: version(name) for name in ("boto3", "botocore", "bedrock-agentcore")})
import workshop_utils

print("Shared package:", workshop_utils.__file__)


## 1. Markdown vs code

You're reading a **Markdown cell** right now. It renders headings, explanations, links, and tables. Double-click it (or use the edit control) to see the source; run it with `Shift + Enter` to render the text again.

The next cell is a **code cell**. Run it with `Shift + Enter`. Python executes the `print()` instruction and places the result below the cell. Editing code changes the document; running it changes the kernel's state.


In [ ]:
print("Hello from Jupyter!")

You should see `Hello from Jupyter!` appear directly below the cell.

<div class="alert alert-block alert-info">

**If the cell did not run**: select **Bedrock Workshop (Python 3.13)** from the notebook kernel selector, then run it again.

</div>

---

## 2. State persists across cells

Variables, imports, function definitions — everything you create stays in the kernel until you restart it. Run the next two cells in order:


In [ ]:
greeting = "Hi from cell 2"
number = 42

Now read the variables from the previous cell. The values are available because both cells use the same kernel.


In [ ]:
# This cell reads the variables defined above
print(greeting)
print(f"The number is: {number}")

Now **overwrite** `number` and print it again. Run the next cell and watch the value change from `42` to `50`:

In [ ]:
# Reassign `number` — the kernel keeps the latest value you give it
number = 50
print(f"The number is now: {number}")

You just saw it: `number` was `42`, the reassignment cell pushed it to `50`, and that new value persists for every cell that runs afterward. **The kernel remembers everything until you restart it.**

This is powerful — but also where most beginner mistakes come from. Skip a cell, and the next one breaks. Edit a function but forget to re-run, and the old version is still active. Re-run a cell that increments a value, and it climbs every time.

---

## 3. Inspect the kernel environment

The next cell shows the Python interpreter and package versions used by this kernel. `%pip show boto3` is an IPython command that reads package information from this environment. A command prefixed with `!` runs in a shell and can use a different Python interpreter.

`%pip` is an IPython *magic*: a notebook command that knows which Python kernel is active. By contrast, `!ls` would ask a shell to list files, and `!python --version` would show whichever Python the shell finds. This distinction explains why a package can appear installed in a terminal yet be unavailable in a notebook.


In [ ]:
# These inspect the running kernel; they do not call AWS or install packages.
import sys
from importlib.metadata import version

print(sys.executable)
print(sys.version)
print("boto3:", version("boto3"))

# IPython magic uses this kernel's Python; it is read-only here.
%pip show boto3

# Installation belongs in the reproducible terminal setup above, not an unpinned !pip cell.


## 4. The two shortcuts that matter

| Shortcut | What it does |
|---|---|
| `Shift + Enter` | Run cell, move to next |
| `Cmd/Ctrl + Enter` | Run cell, stay |

That's it. Run the next cell to see the result of the last expression auto-print:


In [ ]:
# Note: no print() needed — the last expression in a cell auto-displays
2 + 2

## 5. Your first Bedrock call

Now the real check: can this notebook talk to Amazon Bedrock?

The next cell:
1. Imports `boto3` (AWS SDK for Python)
2. Creates a Bedrock runtime client in the workshop Region, `us-east-1`
3. Selects the workhorse from the shared model registry and builds a **Converse API** request with one user message
4. Prints the response, stop reason, and token usage

The default workhorse is Sonnet 5. The request builder applies the selected model's supported settings. The prepared workshop environment supplies the connection settings.

Run the cell, read the short greeting, and locate the input and output token counts. **This is one billed inference request.** The next lesson explains what those counts mean for cost.


In [ ]:
import json
import os

import boto3
from botocore.config import Config
from dotenv import load_dotenv

from workshop_utils.bedrock import build_converse_request
from workshop_utils.models import ModelRegistry, Support
from workshop_utils.pacing import attach_boto3, get_workshop_pacer

load_dotenv(REPO / ".env", override=False)
REGION = "us-east-1"
registry = ModelRegistry(region=REGION)
selected = registry.resolve("workhorse", override=os.getenv("WORKSHOP_MODEL_ID") or None)
print(json.dumps(selected.as_dict(), indent=2))
bedrock = boto3.client(
    "bedrock-runtime", region_name=REGION,
    config=Config(connect_timeout=5, read_timeout=90, retries={"total_max_attempts": 1}),
)
attach_boto3(bedrock, get_workshop_pacer())
request = build_converse_request(
    selected,
    [{"role": "user", "content": [{"text": "Say hello in one short sentence."}]}],
    max_tokens=256,
    effort="low" if selected.capabilities.support("effort") is Support.SUPPORTED else None,
)
response = bedrock.converse(**request)
print("".join(block["text"] for block in response["output"]["message"]["content"] if "text" in block))
print("Stop reason:", response["stopReason"])
print("Usage:", response["usage"])


## 6. Restarting the kernel

Sometimes state gets confusing: you've edited a function but still have its old definition in memory, or run cells out of order. **Restart the kernel, then re-run from the top** to reconstruct a known state.

Try **Kernel → Restart Kernel** (or the notebook's restart button), then run the `greeting` check below. It reports that the variable has disappeared. Re-run the earlier assignment cell, then run the check again: the greeting returns. Restarting clears memory; it does not remove your saved notebook text.

**Restart and Run All** also executes the cells again, including model calls. To practice state recovery, you only need the greeting assignment and check. A restart is also needed after changing the observability backend, because its providers belong to the running Python process.


In [ ]:
# To practice: restart using the notebook menu, then try the next cell.
# greeting will be undefined until you rerun the cell that assigned it.
# You do not need to re-run the Bedrock call just to restore greeting.


Run this check after restarting the kernel. It reports missing state without stopping the notebook; after rerunning the assignment cell, it prints the greeting again.


In [ ]:
# After restart, this catches the expected missing state without breaking Run All.
try:
    print(greeting)
except NameError:
    print("Kernel state was cleared. Re-run the greeting assignment cell.")


## 7. Interrupting a cell

If you've written an infinite loop or a network call is stuck, you don't need to close the browser. The next cell holds a commented-out infinite loop — uncomment it, run the cell, then stop it with the **interrupt (■ stop) button** on the cell's toolbar:


In [ ]:
# Demo (uncomment to try): this cell loops forever so you can practice interrupting it.
# Run it, then click the interrupt button (the ■ stop icon beside the cell, or in the
# notebook toolbar) to stop it. Kernel → Interrupt Kernel does the same thing.
#
# import time
# i = 0
# while True:
#     i += 1
#     if i % 2 == 0:
#         print(f"still running... iteration {i}")
#     time.sleep(0.5)
print("Uncomment the loop above, run the cell, then stop it with the ■ interrupt button beside the cell.")

While the cell is running, Jupyter shows `[*]` or a busy indicator. Use the **■ stop button** or **Kernel → Interrupt Kernel**. The loop stops with `KeyboardInterrupt`; the kernel stays alive and still remembers its variables. Interrupting a network wait may leave a request already accepted by the service running.

**Restart clears state; interrupt stops the current work.** That distinction is useful when a later lab takes longer than expected.

You are ready to use the notebook as an experiment: read the explanation, predict the result, run the cell, and inspect its output. Continue to [Prompts 101](01-prompts-101.ipynb) for tokens, cost, latency, and the Converse API.


## References

- [Jupyter documentation](https://docs.jupyter.org/en/latest/)
- [IPython interactive tutorial](https://ipython.readthedocs.io/en/stable/interactive/tutorial.html)
- [Converse API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html)
